# 🎬 LipSync Studio · Dublagem de vídeo com InfiniteTalk
**Faça qualquer pessoa falar um novo áudio: boca, rosto, cabeça e corpo acompanham a fala.**

Usa o **[InfiniteTalk](https://github.com/MeiGen-AI/InfiniteTalk)** (MeiGen-AI, Apache 2.0), modelo de dublagem *sparse-frame* construído sobre o Wan2.1 14B. Ele não troca só a boca: recria o quadro inteiro seguindo o vídeo original, com expressões e movimentos de cabeça coerentes com o áudio.

- **Entrada:** um vídeo da pessoa (ou uma foto) + o áudio da fala (por exemplo, a dublagem gerada no Matraca Studio).
- **Saída:** MP4 em 480p ou 720p, 25 fps, com o áudio original em alta qualidade.
- **Tudo roda no Colab:** o modelo e as dependências são baixados aqui (Passo 2). Nada é instalado no seu computador.
- **Ambiente isolado:** o InfiniteTalk roda num venv próprio (Python 3.10, PyTorch 2.4.1, xformers, flash-attn), sem conflito com o Colab.
- **Pesos em fp8 (~32 GB):** a versão quantizada oficial cabe na memória da L4 e da A100.
- **Vídeos longos em partes:** cada parte pode ser ouvida e refeita na aba **🧩 Revisão por partes**, sem refazer o resto.
- **Resiliente:** a geração roda em segundo plano. Recarregar a página (F5) não interrompe nada.

> **GPU:** use **L4** (24 GB), **A100** ou **H100** em *Ambiente de execução → Alterar tipo de ambiente de execução*. A T4 não serve (sem bfloat16 e pouca RAM).
>
> **Tempo:** o InfiniteTalk é um modelo de 14B. Conte com vários minutos por parte de 30 s (bem menos na A100/H100). O painel mostra a estimativa em tempo real.

In [ ]:
# @title Passo 1: Instalar a base e baixar o LipSync Studio
# @markdown Instala só o necessário para a interface (FFmpeg, Gradio, uv). O InfiniteTalk tem o próprio ambiente, criado no Passo 2.

!apt-get -y update -qq && apt-get -y install -qq ffmpeg > /dev/null
!pip install -q uv "gradio>=5,<7" requests psutil

# Para testar outra versão, troque a referência (branch, tag ou commit) abaixo.
LIPSYNC_REF = "main"  # @param {type:"string"}
LIPSYNC_REPO = "https://github.com/dgurgel-info/ai-colab-lipsync.git"

import importlib
import os
import shutil
import subprocess
import sys

LIPSYNC_SRC = os.path.abspath("lipsync_src")
if os.path.isdir(LIPSYNC_SRC):
    shutil.rmtree(LIPSYNC_SRC)
clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", LIPSYNC_REF, LIPSYNC_REPO, LIPSYNC_SRC],
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if clone.returncode != 0:
    raise RuntimeError(f"Não foi possível baixar o LipSync Studio ({LIPSYNC_REF}): {clone.stderr.strip()[-300:]}")
if LIPSYNC_SRC not in sys.path:
    sys.path.insert(0, LIPSYNC_SRC)
for name in [m for m in sys.modules if m == "lipsync_studio" or m.startswith("lipsync_studio.")]:
    del sys.modules[name]          # reexecutar a célula carrega o código novo
importlib.invalidate_caches()
import lipsync_studio
print(f"✅ Base instalada. LipSync Studio v{lipsync_studio.__version__} ({LIPSYNC_REF}).")

In [ ]:
# @title Passo 2: Preparar o InfiniteTalk (instala, baixa ~32 GB e carrega o modelo)
# @markdown Na primeira vez leva de 15 a 30 minutos: cria o ambiente, baixa os pesos do Hugging Face e carrega o modelo na memória. Se a célula for executada de novo, só o que faltar é refeito.
MEMORIA_GPU = "Automático"  # @param ["Automático", "Pouca VRAM (L4 / 24 GB)", "Tudo na GPU (A100 / H100)"]

import os
import shutil
import subprocess

import psutil

def _gpu():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                             stdout=subprocess.PIPE, text=True, check=True).stdout.strip().splitlines()[0]
        name, mem = [x.strip() for x in out.split(",")]
        return name, float(mem) / 1024
    except Exception:
        return None, 0.0

GPU_NAME, GPU_VRAM = _gpu()
RAM_GB = psutil.virtual_memory().total / 1e9
DISK_GB = shutil.disk_usage("/content" if os.path.isdir("/content") else ".").free / 1e9
print(f"🖥️ GPU: {GPU_NAME or 'nenhuma'} ({GPU_VRAM:.0f} GB) · RAM: {RAM_GB:.0f} GB · disco livre: {DISK_GB:.0f} GB")
if not GPU_NAME:
    raise RuntimeError("GPU não detectada. Em 'Ambiente de execução > Alterar tipo de ambiente de execução', escolha L4, A100 ou H100.")
if "T4" in GPU_NAME or GPU_VRAM < 20:
    raise RuntimeError(f"A {GPU_NAME} não atende o InfiniteTalk (precisa de bfloat16 e ~20 GB de VRAM). Use L4, A100 ou H100.")
if RAM_GB < 45:
    print("⚠️ Pouca RAM para o modelo de 14B. Se o carregamento falhar, ative 'RAM alta' no tipo de ambiente.")
if DISK_GB < 50:
    print("⚠️ Pouco disco livre: o InfiniteTalk precisa de ~45 GB (pesos + ambiente).")

from lipsync_studio.engine.infinitetalk import InfiniteTalkEngine
from lipsync_studio.jobs import JobManager
from lipsync_studio.studio import Studio

PERSISTENT = {"Automático": -1, "Pouca VRAM (L4 / 24 GB)": 0, "Tudo na GPU (A100 / H100)": 30_000_000_000}[MEMORIA_GPU]
if "engine" in globals():
    engine.stop()                  # reexecução: encerra o worker anterior antes de subir outro
engine = InfiniteTalkEngine(root="/content/lipsync_engine" if os.path.isdir("/content") else "lipsync_engine",
                            persistent=PERSISTENT)

def _progress(pct, text="", until=None):
    print(f"[{pct:3.0f}%] {text}", flush=True)

engine.prepare(log=print, progress=_progress)
studio = Studio(engine, root=os.path.abspath("."))
jobs = JobManager()
print("✅ InfiniteTalk pronto. Execute o Passo 3 para abrir a interface.")

In [ ]:
# @title Passo 3: Abrir a interface (Gradio)
# @markdown Clique no link público `https://...gradio.live` que aparece abaixo. **Não compartilhe o link**: quem tiver o link usa a GPU da sua sessão.

from lipsync_studio.ui.app import build_app, launch

gpu_label = f"{GPU_NAME} · {GPU_VRAM:.0f} GB" if GPU_NAME else None
demo, launch_style = build_app(studio, jobs, gpu_label=gpu_label)
launch(demo, studio, launch_style, share=True, debug=True)

In [ ]:
# @title Passo 4: Baixar todos os vídeos gerados (.ZIP)
# @markdown Compacta a pasta `outputs/` e baixa para o seu computador.

import glob
import os
import zipfile

try:
    from google.colab import files
    in_colab = True
except ImportError:
    in_colab = False

videos = sorted(glob.glob(os.path.join("outputs", "*.mp4")), key=os.path.getmtime, reverse=True)
if not videos:
    print("⚠️ Nenhum vídeo em ./outputs ainda.")
else:
    zip_path = os.path.abspath("videos_dublados.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as zf:
        for path in videos:
            print(f"  🎞️ {os.path.basename(path)} ({os.path.getsize(path) / 1e6:.1f} MB)")
            zf.write(path, os.path.basename(path))
    print(f"\n📦 {len(videos)} vídeo(s) em {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
    if in_colab:
        files.download(zip_path)